# Multi-task PhoBERT: clickbait + topic - 5-fold cross-validation (Kaggle)

**Before running**: Accelerator **GPU**, Internet **On**. Code and processed data are cloned from GitHub (push your latest changes first).

One PhoBERT, two heads on the same feature: clickbait (the task) and the article's topic (13 categories, used only as an extra training signal). Prediction uses the clickbait head alone, so no topic label is needed at prediction time.

Same 5 folds and settings as the text baseline (out-of-fold F1 0.771); the only change is the added topic loss.

| run | loss |
|---|---|
| `topic_0.3` | clickbait loss + 0.3 x topic loss |
| `topic_1.0` | clickbait loss + 1.0 x topic loss |

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_multitask_cv.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_multitask_cv.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/multitask_cv")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "multitask_cv"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Train

About 7 minutes per run on a T4.

In [ ]:
from trainer.train_multitask_cv import MultiTaskConfig, run

BASE = dict(
    data_dir=str(DATA_DIR),
    n_folds=5,
    max_length=128,
    use_lead=True,
    batch_size=32,
    lr=2e-5,
    layer_decay=0.9,
    epochs=5,
    weighted_loss=True,
    patience=3,
    monitor="loss",
    seed=42,
    keep_checkpoints=False,
    limit=0,              # set to e.g. 64 for a quick smoke test
)
RUNS = {
    "topic_0.3": dict(topic_weight=0.3),
    "topic_1.0": dict(topic_weight=1.0),
}

results = {}
for name, overrides in RUNS.items():
    print(f"################ {name} ################")
    results[name] = run(MultiTaskConfig(**BASE, **overrides, out_dir=str(OUT_DIR / name)))

## Compare with the text baseline

Both models predicted the same 3,067 articles out-of-fold, so the difference is tested directly. If the 95% interval includes 0, the topic task does not measurably help.

In [ ]:
import numpy as np
import pandas as pd
from trainer.trainer import classification_metrics

TEXT_DIR = CODE_DIR / "outputs" / "text_baseline"      # PhoBERT, clickbait only, same folds and settings
text = {"oof": pd.read_csv(TEXT_DIR / "oof_predictions.csv"), "test": pd.read_csv(TEXT_DIR / "test_predictions.csv")}

def f1(y, pred):
    tp = (y & pred).sum()
    return 2 * tp / max(2 * tp + (~y & pred).sum() + (y & ~pred).sum(), 1)

rng = np.random.default_rng(0)
rows = {}
for name, r in results.items():
    s = r["summary"]
    for split, key in (("out-of-fold", "oof_predictions"), ("test", "test_predictions")):
        pair = text["oof" if split == "out-of-fold" else "test"][["id", "label", "prob_clickbait"]].merge(
            r[key][["id", "prob_clickbait"]], on="id", suffixes=("_text", "_mt"), validate="one_to_one")
        y = (pair["label"] == "clickbait").to_numpy()
        t, m = (pair["prob_clickbait_text"] >= 0.5).to_numpy(), (pair["prob_clickbait_mt"] >= 0.5).to_numpy()
        idx = rng.integers(0, len(y), (2000, len(y)))
        diffs = np.array([f1(y[i], m[i]) - f1(y[i], t[i]) for i in idx])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        rows[(name, split)] = {"f1": f1(y, m), "text_f1": f1(y, t), "difference": f1(y, m) - f1(y, t),
                               "low": lo, "high": hi, "fixed": int(((t != y) & (m == y)).sum()),
                               "broken": int(((t == y) & (m != y)).sum())}
    rows[(name, "out-of-fold")]["topic_accuracy"] = s.get("oof_topic_accuracy")
comparison = pd.DataFrame(rows).T
display(comparison.round(4))

SHOW = max(results, key=lambda n: results[n]["summary"]["oof"]["f1"])   # run detailed below; set a name to override
result = results[SHOW]
print("showing details for:", SHOW)

### Per topic, out-of-fold

Does the topic task help the topics the text model handles worst (e.g. *Tin tức tổng hợp*, *Chính trị & An ninh*)?

In [ ]:
pair = text["oof"][["id", "category", "label", "prob_clickbait"]].merge(
    result["oof_predictions"][["id", "prob_clickbait", "pred_topic"]], on="id", suffixes=("_text", "_mt"))
per_topic = {}
for topic, g in pair.groupby("category"):
    y = (g["label"] == "clickbait").to_numpy()
    per_topic[topic] = {"n": len(g), "clickbait": int(y.sum()),
                        "text_f1": f1(y, (g["prob_clickbait_text"] >= 0.5).to_numpy()),
                        "multitask_f1": f1(y, (g["prob_clickbait_mt"] >= 0.5).to_numpy()),
                        "topic_head_accuracy": (g["pred_topic"] == topic).mean()}
per_topic = pd.DataFrame(per_topic).T
per_topic["difference"] = per_topic["multitask_f1"] - per_topic["text_f1"]
per_topic.sort_values("n", ascending=False).round(3)

## Per-fold results

`val_*` is each model on its own held-out fold; `test_*` is the same model alone on the test split.

In [ ]:
folds = result["fold_metrics"]
display(folds.round(4))
folds.agg(["mean", "std"]).round(4)

## Cross-validated score vs single model vs ensemble

In [ ]:
summary = pd.DataFrame(result["summary"]).T.loc[["oof", "single_model_test_mean", "single_model_test_std", "ensemble_test"]]
summary.index = ["out-of-fold (pooled train+val)", "single model on test (mean)", "single model on test (std)", "ensemble on test"]
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([f"fold {i}" for i in folds.index], folds["test_f1"], color="#9ecae1", label="single model")
ax.axhline(result["summary"]["ensemble_test"]["f1"], color="#E45756", linestyle="--", label="ensemble")
ax.set_ylim(max(0, folds["test_f1"].min() - 0.1), 1)
ax.set_ylabel("test F1 (clickbait)")
ax.legend(frameon=False)
fig.tight_layout()

## Ensemble on test

In [ ]:
test = result["test_predictions"]
pd.crosstab(test["label"], test["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

### Threshold

Chosen on the out-of-fold predictions (never on test), then applied to the ensemble.

In [ ]:
oof = result["oof_predictions"]
oof_y, test_y = (oof["label"] == "clickbait").to_numpy(), (test["label"] == "clickbait").to_numpy()

rows = []
for t in np.arange(0.2, 0.81, 0.05):
    o = classification_metrics(oof_y, (oof["prob_clickbait"] >= t).to_numpy())
    e = classification_metrics(test_y, (test["prob_clickbait"] >= t).to_numpy())
    rows.append({"threshold": round(t, 2), "oof_precision": o["precision"], "oof_recall": o["recall"], "oof_f1": o["f1"],
                 "test_precision": e["precision"], "test_recall": e["recall"], "test_f1": e["f1"]})
thresholds = pd.DataFrame(rows).set_index("threshold")
print("best threshold on out-of-fold F1:", thresholds["oof_f1"].idxmax())
thresholds.round(3)

### Out-of-fold results per source and category

Computed on all pooled articles, so the groups are large enough to compare.

In [ ]:
def by_group(table, col):
    rows = {}
    for name, g in table.groupby(col):
        m = classification_metrics((g["label"] == "clickbait").to_numpy(), (g["pred"] == "clickbait").to_numpy())
        rows[name] = {"n": len(g), "clickbait_rate": (g["label"] == "clickbait").mean(), **m}
    return pd.DataFrame(rows).T.round(3).sort_values("n", ascending=False)

by_group(oof, "source")

In [ ]:
by_group(oof, "category")

### Most confident out-of-fold mistakes

Each prediction comes from a model that never saw the article, so confident disagreements are good candidates for a label review.

In [ ]:
pd.set_option("display.max_colwidth", 140)
wrong = oof[oof["label"] != oof["pred"]].copy()
wrong["confidence"] = (wrong["prob_clickbait"] - 0.5).abs() + 0.5
print(len(wrong), "out-of-fold mistakes;", (wrong["confidence"] >= 0.9).sum(), "with confidence >= 0.9")
wrong.sort_values("confidence", ascending=False).head(30)[["id", "source", "title", "label", "pred", "prob_clickbait"]]

## Saved files

`summary.json`, `fold_metrics.csv`, `oof_predictions.csv`, `test_predictions.csv`, `tokenizer/`, per-fold `history.json`. On Kaggle they are under `/kaggle/working/multitask_cv/<run name>` and appear in the Output tab after **Save Version**.

In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")